# Sweep vs $L$ — observables & transition
$h_z \\in [0.15, 0.50]$, $L = 6,8,10,12$. Diverged runs skipped.

In [ ]:
import json, re
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
NQS, ED = Path("../results/nqs"), Path("../results/ed")
FIG = Path("../figures"); FIG.mkdir(exist_ok=True)          # high-res outputs land here

In [ ]:
def load(L, hz):
    d = json.load(open(NQS / f"G-equiv_1_L{L}_hx0.00_hz{hz:.3f}.json"))
    E = np.array([complex(x).real for x in d["energy"]])
    V = np.array([complex(x).real for x in d["Vscore"]])
    return d, E, V

def sweep(L):                                   # {hz: data} for finite (non-diverged) runs
    out = {}
    for p in sorted(NQS.glob(f"G-equiv_1_L{L}_hx0.00_hz*.json")):
        hz = float(re.search(r"hz([\d.]+)\.json", p.name).group(1))
        d, E, _ = load(L, hz)
        if np.isfinite(E[-1]):
            out[hz] = d
    return out

Ls = [6, 8, 10, 12]
data = {L: sweep(L) for L in Ls}
cols = dict(zip(Ls, plt.cm.plasma(np.linspace(0, 0.85, len(Ls)))))

mag_z    = lambda d: np.mean(d["order_params"]["magnetization_Zmean"][-1])
wilson_x = lambda d: d["order_params"]["WilsonBFFM"][-1][0] if d["order_params"]["WilsonBFFM"] else np.nan
renyi2   = lambda d: d["order_params"]["renyi2_entropy"][-1][0] if d["order_params"]["renyi2_entropy"] else np.nan

In [ ]:
HZ = 0.30
fig, ax = plt.subplots(figsize=(5.5, 4))
for L in Ls:
    _, _, V = load(L, HZ)
    ax.plot(np.arange(len(V)), V, color=cols[L], lw=1.3, label=f"L={L}")
ax.set(xlabel="step", ylabel="V-score", yscale="log", title=f"learning curves  ($h_z={HZ}$)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "learning_curves_vs_L.png", dpi=300, bbox_inches="tight")

In [ ]:
obs = [(r"$\langle\sigma^z\rangle$", mag_z),
       (r"Wilson $\langle W_X\rangle$", wilson_x),
       ("Rényi $S_2$", renyi2)]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for (label, fn), ax in zip(obs, axes):
    for L in Ls:
        hz = sorted(data[L]); y = [fn(data[L][h]) for h in hz]
        ax.plot(hz, y, "o-", color=cols[L], ms=4, lw=1.2, label=f"L={L}")
    ax.set(xlabel="$h_z$", ylabel=label)
axes[0].legend()
fig.tight_layout()
fig.savefig(FIG / "observables_vs_hz.png", dpi=300, bbox_inches="tight")